In [ ]:
!pip install xgboost


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier


In [ ]:
#Load Dataset
data = pd.read_csv("heart.csv")
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [ ]:
#ML Data Split & Scaling
X = data.drop("target", axis=1)
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
#Tuned XGBoost Model (ML)
ml_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42
)

ml_model.fit(X_train_scaled, y_train)

ml_probs = ml_model.predict_proba(X_test_scaled)[:, 1]

print("ML Accuracy:", accuracy_score(y_test, ml_model.predict(X_test_scaled)))


ML Accuracy: 0.819672131147541


In [ ]:
#Strong NLP Symptom Generation
def generate_symptoms(row):
    symptoms = []

    if row["cp"] in [2, 3]:
        symptoms.append("severe chest pain")
    elif row["cp"] == 1:
        symptoms.append("mild chest pain")

    if row["thalach"] < 120:
        symptoms.append("low heart rate")

    if row["exang"] == 1:
        symptoms.append("exercise induced angina")

    if row["chol"] > 240:
        symptoms.append("high cholesterol")

    if row["oldpeak"] > 2:
        symptoms.append("fatigue dizziness depression")

    if row["age"] > 55:
        symptoms.append("elderly patient")

    return " ".join(symptoms)

data["symptoms"] = data.apply(generate_symptoms, axis=1)
data[["symptoms", "target"]].head()


,symptoms,target
0,severe chest pain fatigue dizziness depression...,1
1,severe chest pain high cholesterol fatigue diz...,1
2,mild chest pain,1
3,mild chest pain elderly patient,1
4,exercise induced angina high cholesterol elder...,1


In [ ]:
#NLP Train-Test Split
X_text = data["symptoms"]

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
#TF-IDF + SVM (NLP Model)
vectorizer = TfidfVectorizer(max_features=1500)
X_text_train_tfidf = vectorizer.fit_transform(X_text_train)
X_text_test_tfidf = vectorizer.transform(X_text_test)

nlp_model = SVC(kernel="linear", probability=True)
nlp_model.fit(X_text_train_tfidf, y_text_train)

nlp_probs = nlp_model.predict_proba(X_text_test_tfidf)[:, 1]

print("NLP Accuracy:", accuracy_score(y_text_test, nlp_model.predict(X_text_test_tfidf)))


NLP Accuracy: 0.7377049180327869


In [ ]:
#STACKING (META-CLASSIFIER)
# Combine ML + NLP probabilities
stacked_X = np.column_stack((ml_probs, nlp_probs))

meta_model = LogisticRegression()
meta_model.fit(stacked_X, y_test)

final_preds = meta_model.predict(stacked_X)


In [ ]:
#FINAL EVALUATION
print("🔥 HYBRID MODEL ACCURACY:")
print(accuracy_score(y_test, final_preds))

print("\n📊 CLASSIFICATION REPORT (HYBRID):")
print(classification_report(y_test, final_preds))


🔥 HYBRID MODEL ACCURACY:
0.8032786885245902

📊 CLASSIFICATION REPORT (HYBRID):
              precision    recall  f1-score   support

           0       0.86      0.68      0.76        28
           1       0.77      0.91      0.83        33

    accuracy                           0.80        61
   macro avg       0.82      0.79      0.80        61
weighted avg       0.81      0.80      0.80        61

